In [32]:
import pandas as pd
import numpy as np

In [33]:
df = pd.read_csv("DateFruit_Dataset.csv")

In [34]:
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [35]:
df.isnull().sum()

AREA             0
PERIMETER        0
MAJOR_AXIS       0
MINOR_AXIS       0
ECCENTRICITY     0
EQDIASQ          0
SOLIDITY         0
CONVEX_AREA      0
EXTENT           0
ASPECT_RATIO     0
ROUNDNESS        0
COMPACTNESS      0
SHAPEFACTOR_1    0
SHAPEFACTOR_2    0
SHAPEFACTOR_3    0
SHAPEFACTOR_4    0
MeanRR           0
MeanRG           0
MeanRB           0
StdDevRR         0
StdDevRG         0
StdDevRB         0
SkewRR           0
SkewRG           0
SkewRB           0
KurtosisRR       0
KurtosisRG       0
KurtosisRB       0
EntropyRR        0
EntropyRG        0
EntropyRB        0
ALLdaub4RR       0
ALLdaub4RG       0
ALLdaub4RB       0
Class            0
dtype: int64

In [36]:
df["Class"].unique()

<StringArray>
['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY']
Length: 7, dtype: str

In [37]:
X = df.drop(columns=["Class"])
y = df["Class"]

In [38]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

le = LabelEncoder()
y = le.fit_transform(y)

In [39]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [40]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ANN

In [41]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,TensorDataset

In [48]:
X_train_tensor = torch.tensor(X_train_scaled,dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test_scaled,dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [49]:
train_dataset = TensorDataset(X_train_tensor,y_train_tensor)
test_dataset = TensorDataset(X_test_tensor,y_test_tensor)

In [50]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32)

In [51]:
# Build Model Blueprint
class ANN(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            # 1 Hidden layer
            nn.Linear(34,64),
            nn.ReLU(),

            # 2 Hidden layer
            nn.Linear(64,64),
            nn.ReLU(),

            # output layer
            nn.Linear(64,7)
        )

    def forward(self,x):
        return self.model(x)

In [52]:
# Build Model
model = ANN()

# Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [53]:
# Training the NN4

epochs = 100
for epoch in range(epochs):
    model.train()

    running_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()

        outputs = model(xb)
        loss = criterion(outputs,yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)

    print(f"epoch = {epoch+1}/{epochs}, loss = {train_loss}")

epoch = 1/100, loss = 1.7407408900882886
epoch = 2/100, loss = 1.166260918845301
epoch = 3/100, loss = 0.7679285731004633
epoch = 4/100, loss = 0.5709187492080356
epoch = 5/100, loss = 0.48653678531232086
epoch = 6/100, loss = 0.4144230135109114
epoch = 7/100, loss = 0.36344535130521527
epoch = 8/100, loss = 0.327600190496963
epoch = 9/100, loss = 0.30380202181961224
epoch = 10/100, loss = 0.2806812278602434
epoch = 11/100, loss = 0.2569084672824196
epoch = 12/100, loss = 0.25651795449464215
epoch = 13/100, loss = 0.23140483284774033
epoch = 14/100, loss = 0.2171703058740367
epoch = 15/100, loss = 0.20658083292453186
epoch = 16/100, loss = 0.19708130670630414
epoch = 17/100, loss = 0.18743208471847617
epoch = 18/100, loss = 0.1847704104755236
epoch = 19/100, loss = 0.1737133948051411
epoch = 20/100, loss = 0.16868318811706876
epoch = 21/100, loss = 0.160019110078397
epoch = 22/100, loss = 0.1542809927917045
epoch = 23/100, loss = 0.15176720366529797
epoch = 24/100, loss = 0.14476657367

In [54]:
# Evaluate
model.eval()

total = 0
correct = 0

with torch.no_grad():
    for xb,yb in test_loader:
        outputs = model(xb)
        _,predicted = torch.max(outputs,1)

        correct += (predicted == yb).sum().item()
        total += yb.size(0)


print("Accuracy", correct/total)
        
        

Accuracy 0.9555555555555556
